In [1]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

In [2]:
input_file = "./2_2_preprocessing of feature labels.xlsx"
df = pd.read_excel(input_file)
df = df.iloc[1:].reset_index(drop=True)  # отбрасываем номера признаков
df.shape

(8452, 311)

In [3]:
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
0,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
1,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
2,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
3,ЧУВ5лмдс_51,5,мужской,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
4,ЧУВ5лмдс_50,5,мужской,NaN,NaN,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8447,ЧУВ5лждс (79),5,женский,NaN,смещение вправо,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,пейзаж,NaN,NaN,солнце,земля,NaN,NaN,NaN,ничего из перечисленного
8448,ЧУВ5лждс (78),5,женский,NaN,NaN,смещение влево,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,просто линия
8449,ЧУВ5лждс (77),5,женский,NaN,смещение вправо,NaN,NaN,смещение вниз,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8450,ЧУВ5лждс (76),5,женский,без особенностей,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,просто линия


In [4]:
statistics = []

for col_idx in range(1, len(df.columns)):
    col_name = df.columns[col_idx]
    column_data = df[col_name]
    
    missing_count = int(column_data.isna().sum())
    
    value_counts = column_data.value_counts(dropna=False)
    
    stat_row = {
        'features': col_name,
        'unique_values': [],
        'frequencies': [],
        'missing_values': missing_count
    }
    
    has_empty = False
    for value, count in value_counts.items():
        value_str = str(value)
        if pd.isna(value) or value_str.lower() == 'nan':
            value_str = 'пусто'
            has_empty = True
        stat_row['unique_values'].append(value_str)
        stat_row['frequencies'].append(int(count))

    if missing_count > 0 and not has_empty:
        stat_row['unique_values'].append('пусто')
        stat_row['frequencies'].append(missing_count)

    if col_name == 'Возраст обследуемого':
        pairs = list(zip(stat_row['unique_values'], stat_row['frequencies']))

        def age_sort_key(item):
            v = item[0]
            if v == 'пусто':
                return (2, 0.0)
            try:
                return (0, float(v))
            except (ValueError, TypeError):
                return (1, str(v))

        pairs.sort(key=age_sort_key)
        stat_row['unique_values'] = [p[0] for p in pairs]
        stat_row['frequencies'] = [p[1] for p in pairs]

    statistics.append(stat_row)

In [5]:
prefix = "3"

In [6]:
analysis_dir = f"./{prefix}_analysis"
os.makedirs(analysis_dir, exist_ok=True)

output_file = f"{analysis_dir}/{prefix}_statistics_analysis.xlsx"

max_unique_values = max(len(stat['unique_values']) for stat in statistics)

wb = Workbook()
ws = wb.active
ws.title = "Статистика признаков"

headers = ['Признак']
for i in range(max_unique_values):
    headers.append(f'Значение_{i+1}')
    headers.append(f'Частота_{i+1}')
headers.append('Пропуски')

cell_font = Font(bold=False)
cell_align = Alignment(horizontal='left', vertical='center')

for col_idx, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.font = cell_font
    cell.alignment = cell_align

for row_idx, stat in enumerate(statistics, 2):
    col_idx = 1
    
    c = ws.cell(row=row_idx, column=col_idx, value=stat['features'])
    c.font = cell_font
    c.alignment = cell_align
    col_idx += 1
    
    unique_values = stat['unique_values']
    frequencies = stat['frequencies']
    
    for i in range(max_unique_values):
        if i < len(unique_values):
            uv = unique_values[i]
            if isinstance(uv, str) and uv != 'пусто':
                try:
                    f = float(uv)
                    if f.is_integer():
                        uv = int(f)
                except ValueError:
                    pass
            c = ws.cell(row=row_idx, column=col_idx, value=uv)
            c.font = cell_font
            c.alignment = cell_align
            col_idx += 1
            c = ws.cell(row=row_idx, column=col_idx, value=int(frequencies[i]))
            c.font = cell_font
            c.alignment = cell_align
            col_idx += 1
        else:
            col_idx += 2
    
    c = ws.cell(row=row_idx, column=col_idx, value=int(stat['missing_values']))
    c.font = cell_font
    c.alignment = cell_align

for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    adjusted_width = min(max_length + 2, 50)
    ws.column_dimensions[col_letter].width = adjusted_width

wb.save(output_file)

In [7]:
reports_dir = os.path.join(analysis_dir, f"{prefix}_reports")
os.makedirs(reports_dir, exist_ok=True)

output_file = f"{reports_dir}/{prefix}_statistics_analysis.xlsx"

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

rows_per_page = 4
cols_per_page = 4
graphs_per_page = rows_per_page * cols_per_page

total_features = len(statistics)
num_pages = math.ceil(total_features / graphs_per_page)

for page in range(num_pages):
    start_idx = page * graphs_per_page
    end_idx = min(start_idx + graphs_per_page, total_features)
    page_features = statistics[start_idx:end_idx]
    
    fig, axes = plt.subplots(rows_per_page, cols_per_page, figsize=(20, 16))
    axes = axes.flatten() if rows_per_page > 1 else [axes] if cols_per_page == 1 else axes
    
    for idx, stat in enumerate(page_features):
        ax = axes[idx]
        
        feature_name = stat['features']; # print(feature_name)
        values = stat['unique_values']; # print(values)
        counts = stat['frequencies']; # print(counts)
        
        if len(values) > 0:
            empty_color = '#CCCCCC'
            non_empty_values = [v for v in values if str(v).lower() != 'пусто']
            num_non_empty = len(non_empty_values)
            
            colors = []
            palette = sns.color_palette("husl", max(num_non_empty, 1))
            palette_idx = 0
            
            for v in values:
                if str(v).lower() == 'пусто':
                    colors.append(empty_color)
                else:
                    colors.append(palette[palette_idx % len(palette)])
                    palette_idx += 1
            
            bars = ax.bar(range(len(values)), counts, 
                         color=colors, 
                         alpha=0.7, edgecolor='black')
            
            ax.set_xticks(range(len(values)))
            labels = []
            for v in values:
                v_str = str(v)
                if v_str.lower() in ['nan', 'missing']:
                    v_str = 'пусто'
                labels.append(v_str[:25] + '...' if len(v_str) > 25 else v_str)
            ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
            ax.set_ylabel('Частота', fontsize=10)
            
            title = feature_name[:70] + '...' if len(feature_name) > 70 else feature_name
            ax.set_title(title, fontsize=9, fontweight='bold', pad=5)
            
            ax.grid(True, alpha=0.3, axis='y')
            
            for i, count in enumerate(counts):
                ax.text(i, count, str(count), ha='center', va='bottom', fontsize=7)
        else:
            ax.text(0.5, 0.5, 'Нет данных', ha='center', va='center', 
                   transform=ax.transAxes, fontsize=12)
            ax.set_title(feature_name[:70] + '...' if len(feature_name) > 70 else feature_name, 
                        fontsize=9, fontweight='bold')
            ax.axis('off')
    
    for idx in range(len(page_features), graphs_per_page):
        axes[idx].axis('off')
    
    plt.suptitle(f'Страница {page + 1} из {num_pages}', 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    
    output_file = f"{reports_dir}/{prefix}_all_feats_distrs_{page + 1}.png"
    plt.savefig(output_file, dpi=200, bbox_inches='tight')
    plt.close()
    
    print(output_file)

./3_analysis\3_reports/3_all_feats_distrs_1.png
./3_analysis\3_reports/3_all_feats_distrs_2.png
./3_analysis\3_reports/3_all_feats_distrs_3.png
./3_analysis\3_reports/3_all_feats_distrs_4.png
./3_analysis\3_reports/3_all_feats_distrs_5.png
./3_analysis\3_reports/3_all_feats_distrs_6.png
./3_analysis\3_reports/3_all_feats_distrs_7.png
./3_analysis\3_reports/3_all_feats_distrs_8.png
./3_analysis\3_reports/3_all_feats_distrs_9.png
./3_analysis\3_reports/3_all_feats_distrs_10.png
./3_analysis\3_reports/3_all_feats_distrs_11.png
./3_analysis\3_reports/3_all_feats_distrs_12.png
./3_analysis\3_reports/3_all_feats_distrs_13.png
./3_analysis\3_reports/3_all_feats_distrs_14.png
./3_analysis\3_reports/3_all_feats_distrs_15.png
./3_analysis\3_reports/3_all_feats_distrs_16.png
./3_analysis\3_reports/3_all_feats_distrs_17.png
./3_analysis\3_reports/3_all_feats_distrs_18.png
./3_analysis\3_reports/3_all_feats_distrs_19.png
./3_analysis\3_reports/3_all_feats_distrs_20.png
